In [19]:
import cmsisdsp as dsp
import os

os.environ.setdefault("MPLCONFIGDIR", ".matplotlib")

from IPython.display import display, Markdown, Math
import matplotlib.pyplot as plt
import numpy as np
import cvxpy as cp
from scipy import signal
from scipy import linalg

import import_ipynb
import buck_converter as bc

def np_to_latex_matrix(name, M, precision=4):
    rows = []
    for row in M:
        row_text = " & ".join(f"{value:.{precision}g}" for value in row)
        rows.append(row_text)

    body = r" \\ ".join(rows)

    return rf"{name} = \begin{{bmatrix}} {body} \end{{bmatrix}}"

$$
A = 
    \begin{bmatrix}
        -\frac{R_L}{L} & -\frac{1}{L} \\
        \frac{1}{C} & 0
    \end{bmatrix}
$$

$$
B_0 = 
    \begin{bmatrix}
        0 & 0 \\
        0 & -\frac{1}{C}
    \end{bmatrix}
$$

$$
B_1 = 
    \begin{bmatrix}
        \frac{1}{L} & 0 \\
        0 & -\frac{1}{C}
    \end{bmatrix}
$$

$$
C = 
    \begin{bmatrix}
        1 & 1
    \end{bmatrix}
$$

In [5]:
A = np.array([
    [-bc.R_L / bc.L, -1 / bc.L],
    [1 / bc.C, 0]
    ])
B_0 = np.array([
    [0, 0],
    [0, -1 / bc.C]
    ])
B_1 = np.array([
    [1 / bc.L, 0],
    [0, -1 / bc.C]
    ])
C = np.array([
    [1, 0],
    [0, 1]
    ])

print("Continuous-time state-space matrices:")
display(Math(np_to_latex_matrix("A", A, 4)))
display(Math(np_to_latex_matrix("B_{0}", B_0, 4)))
display(Math(np_to_latex_matrix("B_{1}", B_1, 4)))
display(Math(np_to_latex_matrix("C", C, 4)))

Continuous-time state-space matrices:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [6]:
T_s = 1 / bc.f_sw / 1e2

A_d = (np.eye(2)) + T_s * A
B_0d = (T_s * B_0)
B_1d = (T_s * B_1)

print("Descrete-time state-space matrices:")
print("A_d:\n", A_d)
display(Math(np_to_latex_matrix("A_{d}", A_d, 6)))
print("B_0d:\n", B_0d)
display(Math(np_to_latex_matrix("B_{0d}", B_0d, 6)))
print("B_1d:\n", B_1d)
display(Math(np_to_latex_matrix("B_{1d}", B_1d, 6)))

Descrete-time state-space matrices:
A_d:
 [[ 0.99998    -0.002     ]
 [ 0.00172414  1.        ]]


<IPython.core.display.Math object>

B_0d:
 [[ 0.          0.        ]
 [ 0.         -0.00172414]]


<IPython.core.display.Math object>

B_1d:
 [[ 0.002       0.        ]
 [ 0.         -0.00172414]]


<IPython.core.display.Math object>

$$
A \, H - H \, A - A - Y_{ 0 } \, C = 0 \\
C \, H = 0 \\
H + H^{ \top } + 2 \, I_{ n } > 0 \\
C \, A \, f + C \, Y_{ 0 } \, C \, f - \alpha \, C \, f = 0
$$
where

$A \in \mathbb{ R }^{ n \times n }$ is known

$C \in \mathbb{ R }^{ k \times n }$ is known

$H \in \mathbb{ R }^{ n \times n }$ is unknown

$Y_0 \in \mathbb{ R }^{ n \times k }$ is unknown

$f \in \mathbb{ R }^{ n }$ is known

$\alpha \in \mathbb{ R }$ is uknown

In [22]:
f = np.array([
    [0],
    [1]
    ])

n = A.shape[0]
k = C.shape[0]
delta = 1e-6

assert A.shape == (n, n)
assert C.shape == (k, n)
assert f.shape == (n, 1)

I = np.eye(n)

H       = cp.Variable((n, n))
Y_0     = cp.Variable((n, k))
alpha   = cp.Variable()

constraints = [
    A @ H - H @ A - A - Y_0 @ C == 0,
    C @ H == 0,
    # Strict LMI approximated as >= eps * I
    H + H.T + 2 * I >> delta * I,
    C @ A @ f + C @ Y_0 @ C @ f - alpha * (C @ f) == 0,
]

# objective = cp.Minimize(
#     cp.sum_squares(H) + cp.sum_squares(Y_0) + cp.square(alpha)
# )
objective = cp.Minimize(0)

# Feasibility problem. Small regularizing objective helps choose a bounded solution.
problem = cp.Problem(objective, constraints)
problem.solve(solver=cp.CLARABEL, verbose=True)

H.value[np.abs(H.value) < 1e-9] = 0
Y_0.value[np.abs(Y_0.value) < 1e-9] = 0
alpha.value[np.abs(alpha.value) < 1e-9] = 0

if problem.status in ["optimal", "optimal_inaccurate"]:
    print("Feasible solution found")
    print("H =")
    print(H.value)
    display(Math(np_to_latex_matrix("H", H.value, 6)))
    print("Y_0 =")
    print(Y_0.value)
    display(Math(np_to_latex_matrix("Y_0", Y_0.value, 6)))
    print("alpha =")
    print(alpha.value)
    display(Math(r"\alpha = {:.6g}".format(alpha.value)))

H = H.value
Y_0 = Y_0.value

(CVXPY) May 29 06:38:58 PM: Your problem has 9 variables, 14 constraints, and 0 parameters.
(CVXPY) May 29 06:38:58 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) May 29 06:38:58 PM: DCP verification time: 0.0003 seconds.
(CVXPY) May 29 06:38:58 PM: Expression tree has 6 nodes.
(CVXPY) May 29 06:38:58 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) May 29 06:38:58 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) May 29 06:38:58 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) May 29 06:38:58 PM: Compiling problem (target solver=CLARABEL).
(CVXPY) May 29 06:38:58 PM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> ExactCone2Cone -> EliminateZeroSized -> ConeMatrixStuffing -> CLARABEL
(CVXPY) May 29 06:38:58 PM: Applying reduction Dcp2Cone
(CVXPY) May 29 06:38:58 PM: Applying reduction CvxAttr2Constr
(CVXPY) May 

                                     CVXPY                                     
                                     v1.9.1                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
-------------------------------------------------------------
           Clarabel.rs v0.11.1  -  Clever Acronym                

                   (c) Paul Goulart                          
                University of Oxford, 2022                   
-------------------------------------------------------------

problem:
  variables     = 9
  constraints  

(CVXPY) May 29 06:38:58 PM: Problem status: optimal
(CVXPY) May 29 06:38:58 PM: Optimal value: 0.000e+00
(CVXPY) May 29 06:38:58 PM: Compilation took 2.553e-02 seconds
(CVXPY) May 29 06:38:58 PM: Solver (including time spent in interface) took 8.530e-03 seconds


  0  +0.0000e+00  -9.5377e+01  9.54e+01  1.84e-16  2.52e-02  1.00e+00  3.21e+01   ------   
  1  +0.0000e+00  -9.5377e-01  9.54e-01  6.84e-17  2.55e-04  1.00e-02  3.21e-01  9.90e-01  
  2  +0.0000e+00  -9.5377e-03  9.54e-03  5.54e-19  2.55e-06  1.00e-04  3.21e-03  9.90e-01  
  3  +0.0000e+00  -9.5377e-05  9.54e-05  5.05e-17  2.55e-08  1.00e-06  3.21e-05  9.90e-01  
  4  +0.0000e+00  -9.5377e-07  9.54e-07  8.41e-17  2.55e-10  1.00e-08  3.21e-07  9.90e-01  
  5  +0.0000e+00  -9.5377e-09  9.54e-09  5.90e-17  2.55e-12  1.00e-10  3.21e-09  9.90e-01  
---------------------------------------------------------------------------------------------
Terminated with status = Solved
solve time = 246.294µs
-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------
Feasible solution found
H =
[[0. 0.]
 [0. 0.]]


<IPython.core.display.Math object>

Y_0 =
[[   20.          2000.        ]
 [-1724.13793103     0.        ]]


<IPython.core.display.Math object>

alpha =
0.0


<IPython.core.display.Math object>

In [23]:
L_0 = np.linalg.inv(H + np.eye(n)) @ Y_0
print("L_0 =")
print(L_0)
display(Math(np_to_latex_matrix("L_0", L_0, 6)))

print(np.square(A + L_0 @ C))

L_0 =
[[   20.          2000.        ]
 [-1724.13793103     0.        ]]


<IPython.core.display.Math object>

[[1.26217745e-29 5.16987883e-26]
 [0.00000000e+00 0.00000000e+00]]


In [24]:
Phi = A + L_0 @ C
M = np.zeros((C.shape[0], n))

v = f

for j in range(n):
    M[:, j] = (C @ v).flatten()
    v = Phi @ v
M[np.abs(M) < 1e-9] = 0

print("M =")
print(M)

print("rank(M) =", np.linalg.matrix_rank(M, tol=1e-9))

M =
[[0. 0.]
 [1. 0.]]
rank(M) = 1


In [26]:
A_0 = A + L_0 @ C

p = p = np.array([-50, -1000])

result = signal.place_poles(A_0.T, C.T, p)

L_FT = -result.gain_matrix.T

A_FT = A + L_0 @ C + L_FT @ C
eig_A_FT = np.linalg.eigvals(A_FT)

print("L_FT =")
print(L_FT)

print("eig(A_FT) =")
print(eig_A_FT)

L_FT =
[[-1.00000000e+03 -2.27373675e-13]
 [ 0.00000000e+00 -5.00000000e+01]]
eig(A_FT) =
[-1000.   -50.]


In [27]:
Phi = A + L_0 @ C

v = L_FT @ C @ f

M = np.zeros((k, n + 1))

x = v
for j in range(n):
    M[:, j] = (C @ x).flatten()
    x = Phi @ x

M[:, n] = (C @ f).ravel()
M[np.abs(M) < 1e-9] = 0

print("M =")
print(M)

print("rank(M) =", np.linalg.matrix_rank(M, tol=1e-9))

M =
[[  0.   0.   0.]
 [-50.   0.   1.]]
rank(M) = 1


$$
\begin{pmatrix}
    P \, (A + L_0 \, C + L_{ FT } \, C) + (A + L_0 \, C + L_{ FT } \, C)^{ \top } \, P + \rho \, P & P \\
    P & -Z
\end{pmatrix}
\leq 0
$$

$$
P > 0, \quad Z > 0, \quad X > 0
$$

$$
\begin{pmatrix}
    \tau \, X & Y^{ \top } \\
    Y & \theta \, P
\end{pmatrix}
\geq 0
$$

$$
P \geq C^{ \top } \, X \, C
$$

$$
\nu \, P \, H + \nu \, H^{ \top } \, P + 2 \, \epsilon \, P > 0
$$

$$
\begin{pmatrix}
    2 \, ( \nu + \epsilon ) \, Z + \nu \, H^{ \top } \, Z + \nu \, Z \, H & \nu \, Z \, ( I_{ n } + H ) \\
    \nu \, ( I_{ n } + H^{ \top } ) \, Z & M 
\end{pmatrix}
\geq 0
$$

$$
P > 0, \quad M > 0, \quad Z > 0 ,
$$

$$
q_{ i }^{ 2 \, \epsilon } \, \Xi^{ \top }( q_{ i } ) \, Z \, \Xi( q_{ i } ) + q_{ i }^{ 2 \, \epsilon - 1 } ( q_{ i } - q_{ i - 1 } ) M \leq \frac{ 1 }{ \theta } \, P, \quad i = \overline{ 1, \ N }
$$

where

$0 = q_{ 0 } < q_{ 1 } < \dots < q_{ N } = 1$

$H, \ P, \ M, \ Z \in \mathbb{ R }^{n \times n}$

$P, \ Z, \in \mathbb{ R }^{ n \times n }$

$X \in \mathbb{ R }^{ k \times k }$

$Y \in \mathbb{ R }^{ n \times k }$

$\Xi( \lambda ) = \exp{ \left( \left( \nu \, H + \nu \, I_{ n } \right) \ln{ \lambda } \right) } - I_{ n }$

In [36]:
nu = -0.5
tau = 1
rho = tau * 1.0001
epsilon = 1.5
theta = 1
q = np.arange(0, 1.0001, 0.025)

n = A.shape[0]
k = C.shape[0]
N = len(q) - 1

A_FT = A + L_0 @ C + L_FT @ C

P = cp.Variable((n, n))
Z = cp.Variable((n, n))
M = cp.Variable((n, n))
X = cp.Variable((k, k))
Y = cp.Variable((n, k))

constraints = []

block_1 = cp.bmat([
    [P @ A_FT + A_FT.T @ P + rho * P,   P],
    [P,                                 -Z]
])
constraints.append(block_1 << 0)

constraints += [
    P >> delta * np.eye(n),
    Z >> delta * np.eye(n),
    X >> delta * np.eye(k),
]

block_2 = cp.bmat([
    [tau * X,      Y.T],
    [Y,            theta * P]
])
constraints.append(block_2 >> 0)

constraints.append(P - C.T @ X @ C >> 0)

constraints.append(
    nu * (P @ H + H.T @ P) + 2 * epsilon * P >> delta * np.eye(n)
)

I = np.eye(n)

block_3 = cp.bmat([
    [
        2 * (nu + epsilon) * Z + nu * (H.T @ Z + Z @ H),    nu * Z @ (I + H)
    ],
    [
        nu * (I + H.T) @ Z,                                 M
    ]
])

constraints.append(block_3 >> 0)

constraints.append(M >> delta * np.eye(n))

for i in range(1, len(q)):
    qi = q[i]
    qim1 = q[i - 1]

    Xi_i = linalg.expm(np.log(qi) * (nu * (H + I))) - I

    lhs = (
        qi ** (2 * epsilon) * (Xi_i.T @ Z @ Xi_i)
        + qi ** (2 * epsilon - 1) * (qi - qim1) * M
    )

    constraints.append(lhs << (1 / theta) * P)

problem = cp.Problem(cp.Minimize(0), constraints)

problem.solve(
    solver=cp.CLARABEL,
    verbose=True
)

if problem.status in ["optimal", "optimal_inaccurate"]:
    print("Feasible solution found")

    print("P =")
    print(np.round(P.value, 6))

    print("Z =")
    print(np.round(Z.value, 6))

    print("M =")
    print(np.round(M.value, 6))

    print("X =")
    print(np.round(X.value, 6))

    print("Y =")
    print(np.round(Y.value, 6))

else:
    print("Problem not solved successfully")

    P = P.value
    Z = Z.value
    M = M.value
    X = X.value
    Y = Y.value

(CVXPY) May 29 06:52:11 PM: Your problem has 20 variables, 232 constraints, and 0 parameters.
(CVXPY) May 29 06:52:11 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) May 29 06:52:11 PM: DCP verification time: 0.0121 seconds.
(CVXPY) May 29 06:52:11 PM: Expression tree has 8 nodes.
(CVXPY) May 29 06:52:11 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)


(CVXPY) May 29 06:52:11 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) May 29 06:52:11 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) May 29 06:52:11 PM: Compiling problem (target solver=CLARABEL).
(CVXPY) May 29 06:52:11 PM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> ExactCone2Cone -> EliminateZeroSized -> ConeMatrixStuffing -> CLARABEL
(CVXPY) May 29 06:52:11 PM: Applying reduction Dcp2Cone
(CVXPY) May 29 06:52:11 PM: Applying reduction CvxAttr2Constr
(CVXPY) May 29 06:52:11 PM: Applying reduction ExactCone2Cone
(CVXPY) May 29 06:52:11 PM: Applying reduction EliminateZeroSized
(CVXPY) May 29 06:52:11 PM: Applying reduction ConeMatrixStuffing


                                     CVXPY                                     
                                     v1.9.1                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) May 29 06:52:11 PM: Applying reduction CLARABEL
(CVXPY) May 29 06:52:11 PM: Finished problem compilation (took 1.929e-01 seconds).
(CVXPY) May 29 06:52:11 PM: Invoking solver CLARABEL  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
-------------------------------------------------------------
           Clarabel.rs v0.11.1  -  Clever Acronym                

                   (c) Paul Goulart                          
                University of Oxford, 2022                   
-------------------------------------------------------------

problem:
  variables     = 20
  constraints   = 168
  nnz(P)        = 0
  nnz(A)        = 544
  cones (total) = 49
    : PSDTriangle = 49,  numel = (10,3,3,3,...,3)

settings:
  linear algebra: direct / qdldl, precision: 64 bit (1 thread)
  max iter = 200, time limit = Inf,  max step = 0.990
  tol_feas = 1.0e-8, tol_gap_abs = 1.0e-8, tol_gap_rel = 1.0e-8,
  static reg : on, ϵ1 = 1.0e-8, ϵ2 = 4.9e-32
  dynamic reg: on, ϵ = 1.0e-13, δ = 2.0e-

(CVXPY) May 29 06:52:11 PM: Problem status: optimal
(CVXPY) May 29 06:52:11 PM: Optimal value: 0.000e+00
(CVXPY) May 29 06:52:11 PM: Compilation took 1.929e-01 seconds
(CVXPY) May 29 06:52:11 PM: Solver (including time spent in interface) took 1.349e-02 seconds


  0  +0.0000e+00  +8.9537e-05  8.95e-05  1.00e+00  9.02e+01  1.00e+00  1.00e+00   ------   
  1  +0.0000e+00  +8.8666e-05  8.87e-05  9.13e-01  4.69e+01  4.72e-01  4.72e-01  6.71e-01  
  2  +0.0000e+00  +6.1692e-05  6.17e-05  6.41e-01  5.39e+01  4.23e-01  4.23e-01  4.21e-01  
  3  +0.0000e+00  +5.0224e-05  5.02e-05  8.68e-02  8.83e+00  6.35e-02  6.35e-02  9.19e-01  
  4  +0.0000e+00  +3.6438e-05  3.64e-05  4.16e-02  6.62e+00  3.59e-02  3.58e-02  7.01e-01  
  5  +0.0000e+00  +1.4749e-05  1.47e-05  1.07e-02  7.95e+00  1.46e-02  1.46e-02  9.90e-01  
  6  +0.0000e+00  +1.6853e-06  1.69e-06  7.27e-04  7.57e+00  1.21e-03  1.21e-03  9.30e-01  
  7  +0.0000e+00  +2.1485e-07  2.15e-07  1.49e-05  6.57e-01  2.62e-05  2.59e-05  9.79e-01  
  8  +0.0000e+00  +2.2026e-09  2.20e-09  1.50e-07  6.67e-03  2.65e-07  2.63e-07  9.90e-01  
  9  +0.0000e+00  +2.2026e-11  2.20e-11  1.50e-09  6.67e-05  2.65e-09  2.63e-09  9.90e-01  
 10  +0.0000e+00  +2.2026e-13  2.20e-13  1.50e-11  6.67e-07  2.65e-11  2.63e-11 

In [38]:
G_d = nu * H + epsilon * np.eye(n)


print(np.linalg.eigvals(A_FT))

# Ptilde = X^(1 / 2) write this expression in python
Ptilde = linalg.sqrtm(X.value)

print("Ptilde =")
print(np.round(Ptilde, 6))

[-1000.   -50.]
Ptilde =
[[ 0.035344 -0.      ]
 [ 0.        0.1185  ]]
